# SuperMAG conductance assimilation
Runs the SMRAI / SuperMAG ensemble assimilation and writes one file per event. This notebook is shared on its own. The input data are downloaded separately and placed next to the notebook before you run it.

## Data

**SuperDARN assimilation (electric field).** Download

Nilam, B. (2026). *SMRAI with SuperDARN Data Assimilation for Substorm Analysis*. Zenodo. https://doi.org/10.5281/zenodo.18857294

and place the SuperDARN data in `SMRAI2.1/`. Each event needs

`SMRAI2.1/substorm_YYYYMMDD/SMRAI2.1_DAssim/ef_est*.dat`

**Solar wind.** Download the solar-wind file for the event and save it as `sw_data/swdataYYYYMMDD.dat` (or `sw_data/modified/swdataYYYYMMDD.dat`).

**SuperMAG.** Download the 60 s SuperMAG file and save it as `SM_data/YYYYMMDD.vec-supermag.60s.rev-0006.ncdf`.

The Hall-conductance model also has to sit next to the notebook: `SMRAI2/modelw099_300.sav`, `SMRAI2/meanw.npy`, and `SMRAI2/imgw.npy`.

## Run

`event_dates` lists the events. The default list is 10 May 2024 and 10 October 2024. A date may be written as `2024-05-10` or `20240510`. Running the notebook from top to bottom writes `outputs/assimilation_20240510.npz` and `outputs/assimilation_20241010.npz`.


In [8]:
# Both events. A date may be written as 2024-05-10 or 20240510.
event_dates = ["2024-05-10", "2024-10-10"]

# SRBF centers and ensemble size. 
n_centers = 1000
srbf_kappa = 223.0
mlat_spiral_min = 50.0
mlat_spiral_max = 90.0

K = 1000
c_std = 0.5
q_std = 0.3
obs_err_std = 200.0
w_init_mean = 1.0
w_init_std = 0.25
w_q_std = 0.08
alpha_tanh_factor = 3.0   
dt_min = 5.0
tau_min = 10.0
loc_d0_deg = 10.0
lambda_cov = 1.5
print_every = 50
rng_seed = 0

# Optional input perturbations. Set a flag False to keep that input unchanged.
apply_sw_perturb = True
sw_perturb_seed = 42
sw_perturb_mode = "multiplicative"
sw_perturb_std = {"by": 0.20, "bz": 0.20, "vsw": 0.05, "n": 0.05}

apply_esn_cond_perturb = True
esn_cond_perturb_seed = 24
esn_cond_perturb_mode = "multiplicative"
esn_cond_perturb_std_frac = 0.10
esn_cond_floor = 1e-6

t_start = 0
t_end = None   # None = last common time step of that event


In [9]:
import datetime
import glob
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from geopack import geopack
from scipy.interpolate import griddata


## Shared model, grid, and basis


In [10]:
def perturb_solar_wind_inputs(data_s_in, std_map, seed=0, mode="multiplicative"):
  
    out = np.array(data_s_in, dtype=float, copy=True)
    rng_sw = np.random.default_rng(seed)
    col_idx = {"by": 1, "bz": 2, "vsw": 3, "n": 4}
    n_time = out.shape[0]
    for key, idx in col_idx.items():
        sigma = float(std_map.get(key, 0.0))
        if sigma <= 0.0:
            continue
        eps = rng_sw.normal(0.0, sigma, size=n_time)
        if mode == "multiplicative":
            out[:, idx] = out[:, idx] * (1.0 + eps)
        elif mode == "additive":
            out[:, idx] = out[:, idx] + eps
        else:
            raise ValueError("sw_perturb_mode must be multiplicative or additive.")
    return out

def perturb_esn_conductivity(esn_xy_in, std_frac=0.10, seed=0, mode="multiplicative", floor=0.0):
    out = np.array(esn_xy_in, dtype=float, copy=True)
    eps = np.random.default_rng(seed).normal(0.0, float(std_frac), size=out.shape)
    if mode == "multiplicative":
        out = out * (1.0 + eps)
    elif mode == "additive":
        out = out + eps * np.nanstd(out)
    else:
        raise ValueError("esn_cond_perturb_mode must be multiplicative or additive.")
    if floor is not None:
        out = np.clip(out, float(floor), None)
    return out

def _dim_size(ds, name):
    sizes = getattr(ds, "sizes", None)
    if sizes is not None and name in sizes:
        return int(sizes[name])
    return int(ds.dims[name])


def _decode_station_ids(id_block):
    a = np.asarray(id_block)
    out = []
    if a.ndim == 2:
        for row in a:
            chars = []
            for c in row:
                if c is None:
                    continue
                s = c.decode("utf-8", "ignore") if isinstance(c, (bytes, np.bytes_)) else str(c)
                if s in ("", "--", "None", "nan"):
                    continue
                chars.append(s)
            out.append("".join(chars).strip())
        return out
    for v in np.ravel(a):
        if isinstance(v, (bytes, np.bytes_)):
            out.append(v.decode("utf-8", "ignore").strip("\x00 ").strip())
        else:
            out.append(str(v).strip("\x00 ").strip())
    return out


def _normalize_event_date(value):
    """Accept 2024-05-10 or 20240510."""
    s = str(value).strip()
    if len(s) == 8 and s.isdigit():
        s = f"{s[:4]}-{s[4:6]}-{s[6:]}"
    datetime.datetime.strptime(s, "%Y-%m-%d")
    return s


In [11]:
flearn = 1e-3
with open("SMRAI2/modelw099_300.sav", "rb") as f:
    modelw = pickle.load(f)
meanw = np.load("SMRAI2/meanw.npy")
imgpcaw = np.load("SMRAI2/imgw.npy")
print("Loaded conductivity model")


Loaded conductivity model


In [12]:
lon = np.array([
    0, 6.188, 10.688, 15.188, 19.688, 24.188, 28.688, 33.188, 37.688, 42.188,
    46.688, 51.188, 55.688, 60.188, 64.688, 69.188, 73.688, 78.188, 82.688, 87.188,
    91.688, 96.188, 100.688, 105.188, 109.688, 114.188, 118.688, 123.188, 127.688, 132.188,
    136.688, 141.188, 145.687, 150.188, 154.688, 159.188, 163.688, 168.187, 172.687, 177.187,
    181.687, 186.187, 190.687, 195.187, 199.687, 204.187, 208.687, 213.187, 217.687, 222.187,
    226.687, 231.187, 235.687, 240.187, 244.687, 249.187, 253.687, 258.187, 262.687, 267.187,
    271.687, 276.187, 280.687, 285.187, 289.687, 294.187, 298.687, 303.187, 307.687, 312.187,
    316.687, 321.187, 325.687, 330.187, 334.687, 339.187, 343.687, 348.187, 360, 360,
])
lat = np.array([
    53.109, 55.172, 57.234, 59.297, 61.359, 63.422, 65.484, 66.783, 67.826, 68.870,
    69.913, 70.957, 72.000, 73.044, 74.087, 75.130, 76.174, 77.217, 78.261, 79.304,
    80.348, 81.391, 82.435, 83.478, 84.522, 85.565, 86.609, 87.652, 88.696, 89.739,
])

nlon, nlat = 72, 20
lon1 = np.linspace(0.0, 2 * np.pi, nlon + 1) * (180 / np.pi)
lat1 = 50.0 + 40.0 * np.arange(nlat) / nlat


def interpolate_conductivity(esn_xy_flat, lat_orig, lon_orig, lat_target, lon_target):
    n_time = esn_xy_flat.shape[0]
    n_lat_orig, n_lon_orig = len(lat_orig), len(lon_orig)
    n_lat_new, n_lon_new = len(lat_target), len(lon_target)
    lat_grid_orig, lon_grid_orig = np.meshgrid(lat_orig, lon_orig, indexing="ij")
    points_orig = np.column_stack([lat_grid_orig.ravel(), lon_grid_orig.ravel()])
    lat_grid_new, lon_grid_new = np.meshgrid(lat_target, lon_target, indexing="ij")
    points_new = np.column_stack([lat_grid_new.ravel(), lon_grid_new.ravel()])
    out = np.empty((n_time, n_lat_new, n_lon_new), dtype=float)
    for t in range(n_time):
        values = esn_xy_flat[t].reshape(n_lat_orig, n_lon_orig).ravel()
        interp_flat = griddata(points_orig, values, points_new, method="linear")
        if np.isnan(interp_flat).any():
            nan_mask = np.isnan(interp_flat)
            interp_flat[nan_mask] = griddata(
                points_orig, values, points_new[nan_mask], method="nearest"
            )
        out[t] = interp_flat.reshape(n_lat_new, n_lon_new)
    return out


In [13]:
PHI_GOLDEN = (1.0 + np.sqrt(5.0)) / 2.0


def saff_kuijlaars_spiral_mlat_mlt(n, mlat_min_deg, mlat_max_deg):
   
    if n < 1:
        raise ValueError("n must be >= 1")
    alpha_min_deg = 90.0 - mlat_max_deg
    alpha_max_deg = 90.0 - mlat_min_deg
    cos_a_hi = np.cos(np.deg2rad(alpha_min_deg))
    cos_a_lo = np.cos(np.deg2rad(alpha_max_deg))
    k = np.arange(n, dtype=float)
    u = (k + 0.5) / n
    cos_alpha = np.clip(cos_a_lo + u * (cos_a_hi - cos_a_lo), -1.0, 1.0)
    alpha = np.arccos(cos_alpha)
    phi = (2.0 * np.pi * (k + 1.0) / PHI_GOLDEN) % (2.0 * np.pi)
    mlat = 90.0 - np.rad2deg(alpha)
    mlt_h = phi / (2.0 * np.pi) * 24.0
    return np.column_stack([mlat, mlt_h])


def _mlat_mlt_to_unit_xyz(mlat_deg, mlt_h):
    phi_rad = (np.asarray(mlt_h) / 24.0) * (2.0 * np.pi)
    mlat_rad = np.deg2rad(np.asarray(mlat_deg))
    cl = np.cos(mlat_rad)
    return cl * np.cos(phi_rad), cl * np.sin(phi_rad), np.sin(mlat_rad)


def srbf_basis_theta_phi(theta_grid, phi_grid, theta_j, phi_j, xkappa):
    obx, oby, obz = _mlat_mlt_to_unit_xyz(theta_grid, phi_grid)
    scx, scy, scz = _mlat_mlt_to_unit_xyz(float(theta_j), float(phi_j))
    cdot = np.clip(obx * scx + oby * scy + obz * scz, -1.0, 1.0)
    return np.exp(xkappa * (cdot - 1.0))


gaussian_centers = saff_kuijlaars_spiral_mlat_mlt(n_centers, mlat_spiral_min, mlat_spiral_max)
theta_grid, lon_grid = np.meshgrid(lat1, lon1[:72], indexing="ij")
phi_grid = lon_grid * (24.0 / 360.0)
lat_grid_2d = theta_grid
mlt_grid_2d = phi_grid

psi_all = np.zeros((n_centers,) + theta_grid.shape)
for j in range(n_centers):
    psi_all[j] = srbf_basis_theta_phi(
        theta_grid, phi_grid, gaussian_centers[j, 0], gaussian_centers[j, 1], srbf_kappa
    )
print("psi_all", psi_all.shape, "kappa", srbf_kappa)


psi_all (1000, 20, 72) kappa 223.0


## Assimilate each event and write its file


In [ ]:
saved_paths = []
t_end_requested = t_end
for event_date in event_dates:
    event_date = _normalize_event_date(event_date)
    t_end = t_end_requested
    date_str = event_date.replace("-", "")
    print("=" * 72)
    print("Event", event_date)

    matches = [
        p for p in (
            f"sw_data/swdata{date_str}.dat",
            f"sw_data/modified/swdata{date_str}.dat",
        )
        if os.path.exists(p)
    ]
    if not matches:
        matches = glob.glob(os.path.join("sw_data", "**", f"swdata{date_str}.dat"), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No swdata{date_str}.dat under sw_data/")
    sw_file = sorted(matches)[0]

    sw_cols = [
        "year", "doy", "hour", "minute",
        "p_or_n_proxy", "bt_or_aux", "flow_speed", "density_raw",
        "bx_gsm", "by_gsm", "bz_gsm",
    ]
    sw_df = pd.read_csv(sw_file, sep=r"\s+", header=None, names=sw_cols)
    sw_time_str = (
        sw_df["year"].astype(int).astype(str)
        + sw_df["doy"].astype(int).astype(str).str.zfill(3)
        + sw_df["hour"].astype(int).astype(str).str.zfill(2)
        + sw_df["minute"].astype(int).astype(str).str.zfill(2)
    )
    sw_df["time"] = pd.to_datetime(sw_time_str, format="%Y%j%H%M", utc=True)
    tto = sw_df["time"].astype("int64").to_numpy() / 1e9

    byp2 = sw_df["by_gsm"].ffill().bfill().to_numpy()
    bzp2 = sw_df["bz_gsm"].ffill().bfill().to_numpy()
    nnp_raw = (sw_df["density_raw"] / 1.0e4).ffill().bfill().to_numpy()
    vvp_raw = sw_df["flow_speed"].ffill().bfill().to_numpy()
    vvp2 = np.log10(np.clip(vvp_raw, 1e-6, None)) - 2.5
    nnp2 = np.log10(np.clip(nnp_raw, 1e-6, None)) - 1.0

    tlt2 = np.zeros(tto.shape[0])
    for ii in range(tto.shape[0]):
        tlt2[ii] = geopack.recalc(float(tto[ii]))

    data_s_base = np.vstack((tlt2, byp2, bzp2, vvp2, nnp2)).T.astype(float)
    print(f"Using local SW file: {sw_file}, samples={data_s_base.shape[0]}")

    if apply_sw_perturb:
        data_s = perturb_solar_wind_inputs(
            data_s_base, sw_perturb_std, seed=sw_perturb_seed, mode=sw_perturb_mode
        )
    else:
        data_s = data_s_base.copy()

    esn_xy = (modelw.predict(data_s) / flearn) @ imgpcaw + meanw
    if apply_esn_cond_perturb:
        esn_xy = perturb_esn_conductivity(
            esn_xy,
            std_frac=esn_cond_perturb_std_frac,
            seed=esn_cond_perturb_seed,
            mode=esn_cond_perturb_mode,
            floor=esn_cond_floor,
        )
    print("esn_xy", esn_xy.shape, float(np.nanmin(esn_xy)), "->", float(np.nanmax(esn_xy)))

    conductivity_interp = interpolate_conductivity(esn_xy, lat, lon[:80], lat1, lon1[:80])

    data_folder = f"SMRAI2.1/substorm_{date_str}/SMRAI2.1_DAssim"
    ef_files = sorted(glob.glob(os.path.join(data_folder, "ef_est*.dat")))
    if not ef_files:
        raise FileNotFoundError(f"No ef_est*.dat in {data_folder}")
    print(f"{len(ef_files)} electric-field files in {data_folder}")

    ey_list, ex_list = [], []
    for ef_path in ef_files:
        ef_raw = np.loadtxt(ef_path)
        ey_list.append(ef_raw[:, 2].reshape(20, 72))
        ex_list.append(ef_raw[:, 3].reshape(20, 72))
    Ey_full = np.stack(ey_list)
    Ex_full = np.stack(ex_list)

    sm_name = f"{date_str}.vec-supermag.60s.rev-0006.ncdf"
    sm_path = next(
        (p for p in (Path("SM_data") / sm_name, Path("../SM_data") / sm_name) if p.is_file()),
        None,
    )
    if sm_path is None:
        raise FileNotFoundError(f"Not found: {sm_name} under SM_data")
    print("Loading:", sm_path.resolve())
    ds = xr.open_dataset(sm_path, engine="netcdf4")

    n_blocks_sm = _dim_size(ds, "block")
    n_stations = _dim_size(ds, "vector")
    n_time_model = min(Ex_full.shape[0], conductivity_interp.shape[0], n_blocks_sm // 5)
    print("n_time_model", n_time_model, "n_stations", n_stations)

    E_x_all = Ex_full[:n_time_model]
    E_y_all = Ey_full[:n_time_model]
    Sigma_H_old_all = conductivity_interp[:n_time_model, :, :72]

    dBn_all = np.full((n_time_model, n_stations), np.nan)
    mlat_all = np.full((n_time_model, n_stations), np.nan)
    mlt_all = np.full((n_time_model, n_stations), np.nan)
    station_ids = np.empty((n_time_model, n_stations), dtype="U16")

    for t in range(n_time_model):
        block_idx = t * 5
        if "dbn_geo" in ds:
            dbn_block = ds["dbn_geo"].isel(block=block_idx).values.astype(float)
        elif "dbn_nez" in ds:
            dbn_block = ds["dbn_nez"].isel(block=block_idx).values.astype(float)
        else:
            raise KeyError("Neither dbn_geo nor dbn_nez is in the SuperMAG file.")
        dbn_block[~np.isfinite(dbn_block)] = np.nan
        dbn_block[np.abs(dbn_block) > 1e5] = np.nan
        if "mlat" in ds:
            mlat_block = ds["mlat"].isel(block=block_idx).values.astype(float)
        elif "mcolat" in ds:
            mlat_block = 90.0 - ds["mcolat"].isel(block=block_idx).values.astype(float)
        else:
            raise KeyError("Neither mlat nor mcolat is in the SuperMAG file.")
        dBn_all[t] = dbn_block
        mlat_all[t] = mlat_block
        mlt_all[t] = ds["mlt"].isel(block=block_idx).values.astype(float)
        if "id" in ds:
            ids_t = _decode_station_ids(ds["id"].isel(block=block_idx).values)
            station_ids[t, : min(n_stations, len(ids_t))] = ids_t[:n_stations]
    ds.close()

    alpha_tanh = float(np.log(alpha_tanh_factor))
    phi = float(np.exp(-dt_min / tau_min))
    rng = np.random.default_rng(rng_seed)
    def inflate_forecast_ensemble(xf_ens, lambda_cov):
     
        lam = float(lambda_cov)
        if lam <= 0.0:
            raise ValueError("lambda_cov must be positive")
        if abs(lam - 1.0) < 1e-15:
            return xf_ens
        xf_mean = np.mean(xf_ens, axis=1, keepdims=True)
        return xf_mean + np.sqrt(lam) * (xf_ens - xf_mean)


    def weights_from_latent(z):
        
        return np.asarray(z, dtype=float)


    def latent_from_weights(w):
      
        return np.asarray(w, dtype=float)


    # ------------------------
    # ETKF update (generic state)
    # ------------------------
    def etkf_update_state(x_f_ens, y_f_ens, y_obs, R_var):
        n_state, Kens = x_f_ens.shape

        xbar = np.mean(x_f_ens, axis=1)
        ybar = np.mean(y_f_ens, axis=1)

        X = (x_f_ens - xbar[:, None]) / np.sqrt(Kens - 1.0)
        Y = (y_f_ens - ybar[:, None]) / np.sqrt(Kens - 1.0)

        d = y_obs - ybar

        Y_Rinv = Y / R_var[:, None]
        S = Y.T @ Y_Rinv

        A = np.eye(Kens) + S
        rhs = Y.T @ (d / R_var)
        w = np.linalg.solve(A, rhs)

        x_a_bar = xbar + X @ w

        evals, evecs = np.linalg.eigh(A)
        evals = np.clip(evals, 1e-12, None)
        T = evecs @ (np.diag(evals ** -0.5) @ evecs.T)

        x_a_ens = x_a_bar[:, None] + (x_f_ens - xbar[:, None]) @ T
        return x_a_ens, x_a_bar


    # ------------------------
    # Required variables
    # ------------------------
    required = ["Sigma_H_old_all", "psi_all", "E_x_all", "dBn_all",
                "mlat_all", "mlt_all", "lat_grid_2d", "mlt_grid_2d",
                "lat1", "lon1", "gaussian_centers"]
    missing = [v for v in required if v not in globals()]
    if missing:
        raise NameError(f"Missing in this notebook: {missing}")

    Sigma_H_old_all = globals()["Sigma_H_old_all"]
    psi_all = globals()["psi_all"]
    E_x_all = globals()["E_x_all"]
    dBn_all = globals()["dBn_all"]
    mlat_all = globals()["mlat_all"]
    mlt_all = globals()["mlt_all"]
    lat_grid_2d = globals()["lat_grid_2d"]
    mlt_grid_2d = globals()["mlt_grid_2d"]
    lat1 = globals()["lat1"]
    lon1 = globals()["lon1"]
    gaussian_centers = globals()["gaussian_centers"]

    n_time_model = Sigma_H_old_all.shape[0]
    nlat, nlon = Sigma_H_old_all.shape[1], Sigma_H_old_all.shape[2]

    if t_end is None:
        t_end = n_time_model - 1
    if not (0 <= t_start <= t_end < n_time_model):
        raise ValueError("t_start/t_end out of bounds for Sigma_H_old_all time axis.")

    n_centers = psi_all.shape[0]
    if int(gaussian_centers.shape[0]) != int(n_centers):
        raise ValueError(
            f"gaussian_centers ({gaussian_centers.shape[0]}) must match psi_all ({n_centers})."
        )

    print("CONFIG learn-w ETKF (unconstrained weights):", {
        "n_centers": int(n_centers),
        "K": int(K),
        "dt_min": float(dt_min),
        "tau_min": float(tau_min),
        "phi": float(phi),
        "c_std": float(c_std),
        "q_std": float(q_std),
        "w_init_mean": float(w_init_mean),
        "w_init_std": float(w_init_std),
        "w_q_std": float(w_q_std),
        "alpha_tanh": float(alpha_tanh),
        "sigma_factor_max": float(np.exp(alpha_tanh)),
        "lambda_cov": float(lambda_cov),
        "sqrt_lambda_cov": float(np.sqrt(lambda_cov)),
    })

    # Precompute nearest interpolation map
    grid_points = np.column_stack([lat_grid_2d.flatten(), mlt_grid_2d.flatten()])

    def map_hall_to_stations_nearest(hall_current_grid, station_mlat, station_mlt):
        grid_values = hall_current_grid.flatten()
        finite_mask = np.isfinite(grid_values)
        gp = grid_points[finite_mask]
        gv = grid_values[finite_mask]
        station_points = np.column_stack([station_mlat, station_mlt])
        return griddata(gp, gv, station_points, method="nearest")


    def localization_weights_mlat_mlt(mlat_grid, mlt_grid, station_mlat, station_mlt, d0_deg):
        """Nearest-station distance in combined mlat / MLT space; returns w in [0,1]."""
        nlat, nlon = mlat_grid.shape
        d_min = np.full((nlat, nlon), np.inf, dtype=float)
        for i in range(len(station_mlat)):
            ms, mh = station_mlat[i], station_mlt[i]
            if not (np.isfinite(ms) and np.isfinite(mh)):
                continue
            dlat = mlat_grid - ms
            dm = np.abs(mlt_grid - mh)
            dm = np.minimum(dm, 24.0 - dm)
            dist = np.sqrt(dlat ** 2 + (dm * 15.0) ** 2)
            d_min = np.minimum(d_min, dist)
        w = np.exp(-0.5 * (d_min / float(d0_deg)) ** 2)
        w = np.where(np.isfinite(d_min), w, 0.0)
        return np.clip(w, 0.0, 1.0)


    # ------------------------
    # Full-time storage (filled wherever assimilation runs)
    # ------------------------
    Sigma_after_full_all = np.full((n_time_model, nlat, nlon), np.nan, dtype=float)
    Sigma_after_local_all = np.full((n_time_model, nlat, nlon), np.nan, dtype=float)
    Sigma_after_blend_all = np.full((n_time_model, nlat, nlon), np.nan, dtype=float)
    w_loc_all = np.full((n_time_model, nlat, nlon), np.nan, dtype=float)

    # Backward-compatible + new diagnostics
    c_mean_all = np.full((n_centers, n_time_model), np.nan, dtype=float)           # effective coefficients
    c_raw_mean_all = np.full((n_centers, n_time_model), np.nan, dtype=float)       # learned c_j mean
    w_mean_all = np.full((n_centers, n_time_model), np.nan, dtype=float)            # learned w_j mean
    z_mean_all = np.full((n_centers, n_time_model), np.nan, dtype=float)            # latent z_j mean

    # Forecast observation-space spread (before ETKF update)
    n_obs_st = int(dBn_all.shape[1])
    y_f_mean_all = np.full((n_time_model, n_obs_st), np.nan, dtype=float)
    y_f_std_all = np.full((n_time_model, n_obs_st), np.nan, dtype=float)
    y_a_mean_all = np.full((n_time_model, n_obs_st), np.nan, dtype=float)  # H(x^a_bar), full Sigma


    # ------------------------
    # Run ETKF in time (augmented state x=[c; w])
    # ------------------------
    c_a_ens = rng.normal(0.0, c_std, size=(n_centers, K))
    z_a_ens = w_init_mean + rng.normal(0.0, w_init_std, size=(n_centers, K))

    # Mean-reversion targets
    c_prior_mean = 0.0
    z_prior_mean = np.full((n_centers, 1), w_init_mean, dtype=float)

    for t in range(t_start, t_end + 1):
        # Leaky memory forecast: retain phi of previous analysis, revert to prior mean.
        c_f_ens = (
            phi * c_a_ens
            + (1.0 - phi) * c_prior_mean
            + rng.normal(0.0, q_std, size=c_a_ens.shape)
        )
        z_f_ens = (
            phi * z_a_ens
            + (1.0 - phi) * z_prior_mean
            + rng.normal(0.0, w_q_std, size=z_a_ens.shape)
        )

        # Covariance inflation: x -> xbar + sqrt(lambda) (x - xbar), then h(x_infl), then ETKF
        c_f_ens = inflate_forecast_ensemble(c_f_ens, lambda_cov)
        z_f_ens = inflate_forecast_ensemble(z_f_ens, lambda_cov)

        y_obs_t = dBn_all[t, :]  # (n_stations,)
        n_stations = y_obs_t.shape[0]
        y_pred_ens = np.full((n_stations, K), np.nan, dtype=float)

        Ex_t = E_x_all[t]
        Sigma_old_t = Sigma_H_old_all[t]

        mlat_stations = mlat_all[t]
        mlt_stations = mlt_all[t]

        for k in range(K):
            c_k = c_f_ens[:, k]
            w_k = weights_from_latent(z_f_ens[:, k])  # unconstrained
            ceff_k = c_k * w_k
            u_k = np.tensordot(ceff_k, psi_all, axes=(0, 0))
            # Smooth bounded correction via tanh(u).
            mult_k = np.exp(alpha_tanh * np.tanh(u_k))

            Sigma_new_grid = Sigma_old_t * mult_k
            H_new_grid = Sigma_new_grid * Ex_t

            y_pred_ens[:, k] = map_hall_to_stations_nearest(H_new_grid, mlat_stations, mlt_stations)

        pred_mean = np.nanmean(y_pred_ens, axis=1)
        y_f_mean_all[t] = pred_mean
        y_f_std_all[t] = np.nanstd(y_pred_ens, axis=1, ddof=1)
        valid_mask = np.isfinite(y_obs_t) & np.isfinite(pred_mean)

        m_valid = int(valid_mask.sum())
        do_print = (print_every <= 1) or ((t - t_start) % print_every == 0) or (t == t_end) or (m_valid < 5)
        if do_print:
            print(f"t={t}: m_valid={m_valid}")

        if m_valid < 5:
            if do_print:
                print("  Skipping update (forecast only).")
            y_a_mean_all[t] = pred_mean
            c_a_ens = c_f_ens
            z_a_ens = z_f_ens
            continue

        y_obs_valid = y_obs_t[valid_mask]
        y_pred_valid = y_pred_ens[valid_mask, :]

        station_means = np.nanmean(y_pred_valid, axis=1)
        nan_in_pred = np.isnan(y_pred_valid)
        y_pred_valid[nan_in_pred] = np.take(station_means, np.where(nan_in_pred)[0])

        R_var = (obs_err_std ** 2) * np.ones(m_valid, dtype=float)

        # Augmented ETKF update
        x_f_ens = np.vstack([c_f_ens, z_f_ens])
        x_a_ens, x_a_mean = etkf_update_state(x_f_ens, y_pred_valid, y_obs_valid, R_var)

        c_a_ens = x_a_ens[:n_centers, :]
        z_a_ens = x_a_ens[n_centers:, :]

        c_a_mean = x_a_mean[:n_centers]
        z_a_mean = x_a_mean[n_centers:]
        w_a_mean = weights_from_latent(z_a_mean)
        ceff_a_mean = c_a_mean * w_a_mean

        u_mean = np.tensordot(ceff_a_mean, psi_all, axes=(0, 0))
        mult_mean = np.exp(alpha_tanh * np.tanh(u_mean))
        Sigma_after_full_t = Sigma_old_t * mult_mean
        H_a_grid = Sigma_after_full_t * Ex_t
        y_a_mean_all[t] = map_hall_to_stations_nearest(H_a_grid, mlat_stations, mlt_stations)

        w_loc = localization_weights_mlat_mlt(
            lat_grid_2d, mlt_grid_2d, mlat_stations, mlt_stations, loc_d0_deg
        )
        Sigma_after_local_t = Sigma_old_t * np.exp(w_loc * u_mean)
        Sigma_after_blend_t = (1.0 - w_loc) * Sigma_old_t + w_loc * Sigma_after_full_t

        Sigma_after_full_all[t] = Sigma_after_full_t
        Sigma_after_local_all[t] = Sigma_after_local_t
        Sigma_after_blend_all[t] = Sigma_after_blend_t
        w_loc_all[t] = w_loc

        # Store diagnostics
        c_mean_all[:, t] = ceff_a_mean
        c_raw_mean_all[:, t] = c_a_mean
        w_mean_all[:, t] = w_a_mean
        z_mean_all[:, t] = z_a_mean

        maxdiff = np.nanmax(np.abs(Sigma_after_full_t - Sigma_old_t))
        if do_print:
            ystd_med = float(np.nanmedian(y_f_std_all[t]))
            print(
                f"  max|after-before|={maxdiff:.3e}; "
                f"w_mean min/med/max={np.nanmin(w_a_mean):.3f}/{np.nanmedian(w_a_mean):.3f}/{np.nanmax(w_a_mean):.3f}; "
                f"y^f spread med={ystd_med:.2f} nT"
            )

    # Window slices for polar plots (same t_start:t_end as above)
    Sigma_old_window = Sigma_H_old_all[t_start:t_end+1].copy()
    Sigma_after_full_window = Sigma_after_full_all[t_start:t_end+1].copy()
    Sigma_after_local_window = Sigma_after_local_all[t_start:t_end+1].copy()
    Sigma_after_blend_window = Sigma_after_blend_all[t_start:t_end+1].copy()
    w_loc_window = w_loc_all[t_start:t_end+1].copy()
    Sigma_after_window = Sigma_after_full_window
    c_mean_window = c_mean_all[:, t_start:t_end+1].copy()
    w_mean_window = w_mean_all[:, t_start:t_end+1].copy()

    # Package results for plotting
    res = {
        "Sigma_old_window": Sigma_old_window,
        "Sigma_after_window": Sigma_after_window,
        "Sigma_after_full_window": Sigma_after_full_window,
        "Sigma_after_local_window": Sigma_after_local_window,
        "Sigma_after_blend_window": Sigma_after_blend_window,
        "w_loc_window": w_loc_window,
        "loc_d0_deg": loc_d0_deg,
        "dt_min": dt_min,
        "tau_min": tau_min,
        "phi": phi,
        "alpha_tanh": alpha_tanh,
        "relative_factor_min": float(np.exp(-alpha_tanh)),
        "relative_factor_max": float(np.exp(alpha_tanh)),
        "c_mean_window": c_mean_window,
        "w_mean_window": w_mean_window,
        "t_start": t_start,
        "t_end": t_end,
        "Sigma_after_full_all": Sigma_after_full_all,
        "Sigma_after_local_all": Sigma_after_local_all,
        "Sigma_after_blend_all": Sigma_after_blend_all,
        "w_loc_all": w_loc_all,
        "c_mean_all": c_mean_all,
        "c_raw_mean_all": c_raw_mean_all,
        "w_mean_all": w_mean_all,
        "z_mean_all": z_mean_all,
        "y_f_mean_all": y_f_mean_all,
        "y_f_std_all": y_f_std_all,
        "y_a_mean_all": y_a_mean_all,
        "lon1": lon1,
        "lat1": lat1,
    }

    finite_after = [i for i in range(Sigma_after_window.shape[0]) if np.isfinite(Sigma_after_window[i]).any()]
    print("finite_after idx (last window):", finite_after)
    print("assimilated times (count):", int(np.isfinite(Sigma_after_blend_all).any(axis=(1, 2)).sum()))

    out_dir = Path("outputs")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"assimilation_{date_str}.npz"
    np.savez_compressed(
        out_path,
        event_date=np.array(event_date),
        lat1=lat1,
        lon1=lon1[:72],
        gaussian_centers=gaussian_centers,
        Sigma_H_old_all=Sigma_H_old_all,
        Sigma_after_full_all=Sigma_after_full_all,
        Sigma_after_local_all=Sigma_after_local_all,
        Sigma_after_blend_all=Sigma_after_blend_all,
        w_loc_all=w_loc_all,
        E_x_all=E_x_all,
        E_y_all=E_y_all,
        dBn_all=dBn_all,
        mlat_all=mlat_all,
        mlt_all=mlt_all,
        station_ids=station_ids,
        y_f_mean_all=y_f_mean_all,
        y_f_std_all=y_f_std_all,
        y_a_mean_all=y_a_mean_all,
        c_mean_all=c_mean_all,
        c_raw_mean_all=c_raw_mean_all,
        w_mean_all=w_mean_all,
        z_mean_all=z_mean_all,
        K=np.array(K),
        n_centers=np.array(n_centers),
        srbf_kappa=np.array(srbf_kappa),
        obs_err_std=np.array(obs_err_std),
        lambda_cov=np.array(lambda_cov),
        alpha_tanh=np.array(alpha_tanh),
        dt_min=np.array(dt_min),
        tau_min=np.array(tau_min),
    )
    print("Wrote", out_path.resolve(), "size_MB", round(out_path.stat().st_size / 1e6, 2))
    saved_paths.append(out_path)

print("Finished", len(saved_paths), "events:")
for p in saved_paths:
    print(" ", p)


Event 2024-05-10
Using local SW file: sw_data/modified/swdata20240510.dat, samples=288
esn_xy (288, 2400) 1e-06 -> 53.95040900029802
288 electric-field files in SMRAI2.1/substorm_20240510/SMRAI2.1_DAssim
Loading: /Volumes/AGI PSSD/SMRAI_SM_FAC/SM_data/20240510.vec-supermag.60s.rev-0006.ncdf
n_time_model 288 n_stations 205
CONFIG learn-w ETKF (unconstrained weights): {'n_centers': 1000, 'K': 1000, 'dt_min': 5.0, 'tau_min': 10.0, 'phi': 0.6065306597126334, 'c_std': 0.5, 'q_std': 0.3, 'w_init_mean': 1.0, 'w_init_std': 0.25, 'w_q_std': 0.08, 'alpha_tanh': 1.0986122886681098, 'sigma_factor_max': 3.0000000000000004, 'lambda_cov': 1.5, 'sqrt_lambda_cov': 1.224744871391589}
t=0: m_valid=203
  max|after-before|=6.569e+00; w_mean min/med/max=0.977/1.000/1.028; y^f spread med=5.67 nT
